# nirs_neural_efficiency.ipynb\n\n**Purpose:** Compute the Neural Efficiency (NE) index for each participant × session × ROI\nby merging first-level fNIRS beta coefficients (HbO) with behavioral reading-time data.\n\n**Inputs:**\n- subjstats_completo.xlsx — fNIRS GLM betas exported from MATLAB SubjStats (14 subjects × 10 sessions × 25 channels × 4 chromophores)\n- sessao_stats.xlsx — behavioral data: VELmed (reading speed, ms/char) and TRmed (median response time, s)\n\n**Outputs:**\n- nirs_neural_efficiency.xlsx — 252 rows (14 subjects × 9 training sessions × 2 ROIs) with NE index\n\n**Method:**\n1. Filter HbO channels assigned to left frontal (7 channels) or left temporal (9 channels) ROI.\n2. Average beta across channels within each ROI per subject × session.\n3. z-score eta separately per ROI; z-score VELmed over the full training sample.\n4. NE = (−z_VELmed − z_beta) / √2 (Curtin & Ayaz, 2019).\n   Negative sign on VELmed because lower reading time = faster = better.\n\n**Reference:** Curtin A, Ayaz H (2019). The Age of Neuroergonomics: Towards Ubiquitous and Continuous\nMeasurement of Brain Function with fNIRS. *Jpn Psychol Res*, 61(3), 182-200.\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# fNIRS Neural Efficiency Pipeline
**Project SESI — Z-score calculation and Neural Efficiency Index**

Inputs:
- `subjstats_all.xlsx` — fNIRS SubjStats exported from MATLAB (14000 rows)
- `sessao_stats.xlsx` — behavioral data (wide format, VELmed0–9, TRmed0–9)

Output:
- `fnirs_neural_efficiency.xlsx` — merged table with z-scores and NE index

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

## 1 — Configuration

Canal exluido
'S6-D5': ('P7-P9', 'L Inferior Temporal Gyrus (29%)', 'TEMPORAL'),

In [ ]:
# ── File paths ────────────────────────────────────────────────
# Update this path to match your local data directory
PATH_FNIRS = r'../data/subjstats_completo.xlsx'
# Update this path to match your local data directory
PATH_BEH   = r'../data/sessao_stats.xlsx'
# Update this path to match your local data directory
PATH_OUT   = r'../data/fnirs_neural_efficiency.xlsx'

# ── ROI channel lists ─────────────────────────────────────────
ROI_FRONTAL  = ['S1-D1','S2-D1','S2-D3','S1-D2','S3-D3','S4-D1','S3-D1']
ROI_TEMPORAL = ['S8-D7','S8-D4','S7-D7','S7-D4','S7-D6',
                'S6-D4','S5-D5','S5-D6','S5-D4']

## 2 — Load and prepare fNIRS

In [ ]:
df_fnirs = pd.read_excel(PATH_FNIRS)

df_fnirs['beta'] = df_fnirs['beta'].astype(str).str.replace(',', '.').astype(float)
# Keep only HbO
df_hbo = df_fnirs[df_fnirs['type'] == 'hbo'].copy()

# Assign ROI
def assign_roi(ch):
    if ch in ROI_FRONTAL:  return 'FRONTAL'
    if ch in ROI_TEMPORAL: return 'TEMPORAL'
    return 'OTHER'

df_hbo['roi'] = df_hbo['channel'].apply(assign_roi)
df_hbo = df_hbo[df_hbo['roi'] != 'OTHER'].copy()

# Numeric subject key for merge (SUBJ_002 → 2)
df_hbo['subj_num'] = df_hbo['subject'].str.extract(r'(\d+)').astype(int)

# ── Baseline (calibration = sessao_num 0) ─────────────────────
df_hbo_base = df_hbo[df_hbo['sessao_num'] == 0].copy()
df_hbo_base_agg = (
    df_hbo_base
    .groupby(['subj_num', 'roi'])['beta']
    .mean()
    .reset_index()
    .rename(columns={'beta': 'beta_baseline'})
)

# ── Training sessions (ac_01–ac_09 = sessao_num 1–9) ──────────
df_hbo_train = df_hbo[df_hbo['sessao_num'] >= 1].copy()
df_hbo_agg = (
    df_hbo_train
    .groupby(['subj_num', 'subject', 'group', 'sessao_num', 'roi'])['beta']
    .mean()
    .reset_index()
)

print(f'fNIRS training rows: {len(df_hbo_agg)}')
print(f'fNIRS baseline rows: {len(df_hbo_base_agg)}')

## 3 — Load and prepare behavioral data

In [ ]:
df_beh = pd.read_excel(PATH_BEH)
df_fnirs['beta'] = df_fnirs['beta'].astype(str).str.replace(',', '.').astype(float)

# Numeric subject key (subj001 → 1)
df_beh['subj_num'] = df_beh['SUBJID'].str.extract(r'(\d+)').astype(int)

# Melt VELmed and TRmed columns wide → long
vel_cols = [c for c in df_beh.columns if c.startswith('VELmed')]
tr_cols  = [c for c in df_beh.columns if c.startswith('TRmed')]

df_vel = df_beh.melt(
    id_vars=['subj_num', 'grupo'],
    value_vars=vel_cols,
    var_name='vel_col', value_name='VELmed'
)
df_vel['sessao_num'] = df_vel['vel_col'].str.extract(r'(\d+)').astype(int)

df_tr = df_beh.melt(
    id_vars=['subj_num'],
    value_vars=tr_cols,
    var_name='tr_col', value_name='TRmed'
)
df_tr['sessao_num'] = df_tr['tr_col'].str.extract(r'(\d+)').astype(int)

# Merge VELmed and TRmed
df_beh_long = pd.merge(
    df_vel[['subj_num', 'grupo', 'sessao_num', 'VELmed']],
    df_tr[['subj_num', 'sessao_num', 'TRmed']],
    on=['subj_num', 'sessao_num']
)

# ── Baseline behavioral (sessao_num == 0) ─────────────────────
df_beh_base = (
    df_beh_long[df_beh_long['sessao_num'] == 0]
    [['subj_num', 'VELmed', 'TRmed']]
    .rename(columns={'VELmed': 'VELmed_baseline', 'TRmed': 'TRmed_baseline'})
)

# ── Training behavioral (sessao_num 1–9) ──────────────────────
df_beh_train = df_beh_long[df_beh_long['sessao_num'] >= 1].copy()

print(f'Behavioral training rows: {len(df_beh_train)}')
print(f'Behavioral baseline rows: {len(df_beh_base)}')

## 4 — Merge fNIRS + behavioral

In [ ]:
# Main merge: training data
df = pd.merge(
    df_hbo_agg,
    df_beh_train[['subj_num', 'sessao_num', 'VELmed', 'TRmed']],
    on=['subj_num', 'sessao_num'],
    how='left'
)

# Add HbO baseline
df = pd.merge(df, df_hbo_base_agg, on=['subj_num', 'roi'], how='left')

# Add behavioral baseline
df = pd.merge(df, df_beh_base, on='subj_num', how='left')

# Report unmatched rows
n_missing_vel = df['VELmed'].isna().sum()
print(f'Rows missing VELmed after merge: {n_missing_vel}')
print(f'Total rows: {len(df)}')

## 5 — Z-score calculation

- `z_beta`: computed **per ROI** (FRONTAL and TEMPORAL have separate scales)
- `z_VELmed` / `z_TRmed`: computed once over the full training sample

In [ ]:
# z_beta — per ROI to avoid mixing frontal and temporal scales
df['z_beta'] = (
    df.groupby('roi')['beta']
    .transform(lambda x: zscore(x, nan_policy='omit'))
)

# z_VELmed and z_TRmed — over full training sample
# Note: VELmed is reading speed; lower value = faster = better performance
# Sign is handled in the NE formula, not here
df['z_VELmed'] = zscore(df['VELmed'].dropna().reindex(df.index), nan_policy='omit')
df['z_TRmed']  = zscore(df['TRmed'].dropna().reindex(df.index),  nan_policy='omit')

print('Z-scores calculated.')
print(df[['roi', 'beta', 'z_beta', 'VELmed', 'z_VELmed', 'TRmed', 'z_TRmed']].describe().round(3))

## 6 — Neural Efficiency Index

Formula: Curtin & Ayaz (2019)

`NE = (-z_VELmed - z_beta) / √2`

Negative sign on VELmed because lower reading time = better performance.

In [ ]:
df['neural_efficiency'] = (-df['z_VELmed'] - df['z_beta']) / np.sqrt(2)

print('Neural Efficiency calculated.')
print(df.groupby(['group', 'roi'])['neural_efficiency'].describe().round(3))

## 7 — Export

In [ ]:
cols_out = [
    'subject', 'group', 'sessao_num', 'roi',
    'beta',    'z_beta',
    'VELmed',  'z_VELmed',
    'TRmed',   'z_TRmed',
    'neural_efficiency',
    'beta_baseline', 'VELmed_baseline', 'TRmed_baseline'
]

df_out = df[cols_out].sort_values(['subject', 'sessao_num', 'roi']).reset_index(drop=True)
df_out.to_excel(PATH_OUT, index=False)

print(f'\n✅ File saved: {PATH_OUT}')
print(f'   Shape: {df_out.shape}')
print(f'   Subjects: {df_out["subject"].nunique()}')
print(f'   Sessions: {sorted(df_out["sessao_num"].unique())}')
print(f'   ROIs: {df_out["roi"].unique().tolist()}')
print(f'   NE range: [{df_out["neural_efficiency"].min():.3f}, {df_out["neural_efficiency"].max():.3f}]')
display(df_out.head(6))